In [6]:
import site
cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)

import os, shutil, stat

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/kaggle/working/ptxas-blackwell"

shutil.copy2(src, dst)
os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst


In [7]:
!echo $TRITON_PTXAS_BLACKWELL_PATH

/kaggle/working/ptxas-blackwell


In [8]:
!pip install -q --no-index --find-links /kaggle/input/datasets/zosov07/trl-offline/trl_packages trl

In [9]:
import polars as pl

train = pl.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')

train.head()

id,prompt,answer
str,str,str
"""00066667""","""In Alice's Wonderland, a secre…","""10010111"""
"""000b53cf""","""In Alice's Wonderland, a secre…","""01000011"""
"""00189f6a""","""In Alice's Wonderland, secret …","""cat imagines book"""
"""001b24c4""","""In Alice's Wonderland, numbers…","""XXXVIII"""
"""001c63cb""","""In Alice's Wonderland, secret …","""wizard creates secret"""


In [10]:
# Prepare data
from datasets import Dataset

# Convert to HuggingFace Dataset
train_dict = {
    "prompt": train["prompt"].to_list(),
    "solution": train["answer"].to_list(),  # renamed for accuracy_reward
}

dataset = Dataset.from_dict(train_dict)

# Transform prompt strings into chat format (list of message dicts)
def format_prompt(example):
    example["prompt"] = [{"role": "user", "content": example["prompt"]}]
    return example

dataset = dataset.map(format_prompt)
dataset

Map:   0%|          | 0/9500 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'solution'],
    num_rows: 9500
})

In [11]:
import site

cutlass_pkg_path = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages/"
site.addsitedir(cutlass_pkg_path)

import kagglehub
import mamba_ssm
import torch
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

# Configuration
# MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
MODEL_PATH = "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1"
OUTPUT_DIR = "/kaggle/working"
LORA_RANK = 32  # Can be set to a maximum of 32

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.bfloat16
)
# tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
print("Model loaded successfully.")
model

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Model loaded successfully.


NemotronHForCausalLM(
  (backbone): NemotronHModel(
    (embeddings): Embedding(131072, 2688)
    (layers): ModuleList(
      (0): NemotronHBlock(
        (norm): NemotronHRMSNorm()
        (mixer): NemotronHMamba2Mixer(
          (act): SiLUActivation()
          (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144)
          (in_proj): Linear(in_features=2688, out_features=10304, bias=False)
          (norm): MambaRMSNormGated()
          (out_proj): Linear(in_features=4096, out_features=2688, bias=False)
        )
      )
      (1): NemotronHBlock(
        (norm): NemotronHRMSNorm()
        (mixer): NemotronHMOE(
          (experts): ModuleList(
            (0-127): 128 x NemotronHMLP(
              (up_proj): Linear(in_features=2688, out_features=1856, bias=False)
              (down_proj): Linear(in_features=1856, out_features=2688, bias=False)
              (act_fn): ReLUSquaredActivation()
            )
          )
          (gate): NemotronHTopk

In [12]:
# Total parameters
total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total:,}")

# Trainable vs frozen
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

Total parameters: 31,577,937,344
Trainable parameters: 31,577,937,344


In [13]:
# Initialize LoRA Adapter
print(f"Initializing LoRA adapter with rank={LORA_RANK}...")
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=16,
    target_modules=r".*\.(in_proj|out_proj|up_proj|down_proj)$",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Initializing LoRA adapter with rank=32...


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 880,138,240 || all params: 32,458,075,584 || trainable%: 2.7116


In [ ]:
for example in dataset:
    print(example)
    print(tokenizer.apply_chat_template(example["prompt"], tokenize=True, add_generation_prompt=True))
    break

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# Tokenize with chat template
tokenized_chat_prompts = [tokenizer.apply_chat_template(example["prompt"], tokenize=True, add_generation_prompt=True) for example in dataset]

# Check token lengths across your dataset
lengths = [len(tokenized['input_ids']) for tokenized in tokenized_chat_prompts ]

print(f"Max: {max(lengths)}")
print(f"Min: {min(lengths)}")
print(f"Mean: {sum(lengths)/len(lengths):.0f}")
print(f"Over 512: {sum(1 for l in lengths if l > 512)} / {len(lengths)}")

In [ ]:
# train_grpo.py
from trl import GRPOTrainer, GRPOConfig
from trl.rewards import accuracy_reward

# Configure training
config = GRPOConfig(
    output_dir="grpo_output",
    
    # Learning parameters optimized for reasoning tasks
    learning_rate=3e-6, # Try 5e-6 too

    # Memory-efficient batch configuration
    # Effective batch size = per_device_train_batch_size * num_generations
    # Effective batch size without consuming memory = per_device_train_batch_size * num_generations * gradient_accumulation_steps
    per_device_train_batch_size = 16,
    num_generations=4,          # how many completions per prompt
    gradient_accumulation_steps= 1, 
    
    

    # Sequence length limits for the problems
    max_completion_length=7680,  # adjust based on your answer lengths

    # Sampling and RL training
    temperature=1.0,
    epsilon=10.0,

    # Training duration and monitoring
    max_steps=10,                    # Short demo run (increase to 500+ for production)
    logging_steps=1                 # Log metrics every step for close monitoring
)

# model is your pre-loaded model variable
trainer = GRPOTrainer(
    model=model,
    reward_funcs=accuracy_reward,
    train_dataset=dataset,
    args=config,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


KeyboardInterrupt: 

In [16]:
!ls /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin

cuobjdump  nvdisasm  ptxas  ptxas-blackwell


In [ ]:
# Save Adapter
print(f"Saving adapter to {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)

In [ ]:
import subprocess

subprocess.run("zip -m submission.zip *", shell=True, check=True)

In [ ]:
print('Done.')